# Mars · 02 — First-meet channel-head pairs

**Primary interface** for the first-meet pairing step. This notebook calls the
shared `channel_heads.pairing` package (the same functions the batch wrapper
`scripts/extract_mars_first_meet_pairs.py` uses) — no duplicated logic.

It is **read-only**: it reads the topology GeoPackage and computes pairs *in
memory*. It writes nothing — the full-dataset GeoPackage
(`data/Mars/topology/mars_vn_pairs.gpkg`) is produced by the batch wrapper.

In [1]:
import geopandas as gpd

from channel_heads.io.paths import PROJECT_ROOT
from channel_heads.pairing import (
    build_directed_adjacency,
    chain_segment_geometries,
    first_meet_pairs_on_dag,
    trace_downstream_path,
)

TOPOLOGY_GPKG = PROJECT_ROOT / "data/Mars/topology/mars_vn_topology_model_ready.gpkg"
print("topology:", TOPOLOGY_GPKG, "exists:", TOPOLOGY_GPKG.exists())

topology: /Users/guypi/Projects/channel-heads/data/Mars/topology/mars_vn_topology_model_ready.gpkg exists: True


## Load topology (read-only)

In [2]:
segments = gpd.read_file(TOPOLOGY_GPKG, layer="mars_segments")
nodes = gpd.read_file(TOPOLOGY_GPKG, layer="mars_nodes")
print(f"{segments['network_id'].nunique()} networks, "
      f"{len(segments)} segments, {len(nodes)} nodes")

391 networks, 5619 segments, 6003 nodes


## Pairing for one network — via the package

Pick a network with at least two channel heads, build the directed adjacency,
and run the graph-agnostic first-meet core. `node_id` is local (`0..n-1`) per
network, exactly as the batch wrapper assumes.

In [3]:
# choose a sample network that actually has >=2 channel heads
heads_per_net = (
    nodes[nodes["node_type"] == "channel_head"].groupby("network_id").size()
)
nid = int(heads_per_net[heads_per_net >= 2].index[0])

segs_in_net = segments[segments["network_id"] == nid]
nodes_in_net = nodes[nodes["network_id"] == nid]
n_nodes = int(nodes_in_net["node_id"].max()) + 1

parents, children, edge_to_seg = build_directed_adjacency(segs_in_net, n_nodes)
nt_by_id = dict(zip(nodes_in_net["node_id"], nodes_in_net["node_type"]))
heads_set = {int(i) for i, t in nt_by_id.items() if t == "channel_head"}
confluences_set = {int(i) for i, t in nt_by_id.items() if t == "confluence"}

pairs, _ = first_meet_pairs_on_dag(parents, set(range(n_nodes)), heads_set, confluences_set)
n_pairs = sum(len(s) for s in pairs.values())
print(f"network {nid}: {len(heads_set)} heads, {len(confluences_set)} confluences "
      f"-> {n_pairs} first-meet pair(s) at {len([cc for cc, s in pairs.items() if s])} confluence(s)")

network 1: 27 heads, 26 confluences -> 351 first-meet pair(s) at 26 confluence(s)


## Trace + chain one pair's head→confluence geometry

In [4]:
seg_geom = dict(zip(segs_in_net["segment_id"], segs_in_net.geometry))
start_by = dict(zip(segs_in_net["segment_id"], segs_in_net["start_node_id"]))
end_by = dict(zip(segs_in_net["segment_id"], segs_in_net["end_node_id"]))

conf_id = next(cc for cc, s in pairs.items() if s)
h1, h2 = sorted(pairs[conf_id])[0]
max_steps = n_nodes + 1
for branch, h in [("A", h1), ("B", h2)]:
    path = trace_downstream_path(int(h), int(conf_id), children, max_steps)
    line, seg_ids = chain_segment_geometries(path, edge_to_seg, seg_geom, start_by, end_by)
    print(f"branch {branch}: head {h} -> confluence {conf_id} | "
          f"{len(path)} nodes, {len(seg_ids)} segs, length={line.length:.1f} m")

branch A: head 3 -> confluence 5 | 2 nodes, 1 segs, length=3751.4 m
branch B: head 4 -> confluence 5 | 2 nodes, 1 segs, length=2745.2 m


---
Full-dataset run (writes the GeoPackage all downstream steps consume):

```bash
python scripts/extract_mars_first_meet_pairs.py
```